# CRNN-v2 handwritten circuit-value OCR

This notebook uses only the packaged training and held-out-writer validation data. It does not use the frozen 42-image final test set or any LLM. Select a GPU accelerator and choose **Run All**.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, zipfile

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')
config_candidates = list(INPUT_ROOT.rglob('configs/ocr_crnn_hand_v2.json'))
if not config_candidates:
    zip_candidates = list(INPUT_ROOT.rglob('ocr_crnn_hand_v2_kaggle.zip'))
    if not zip_candidates:
        raise FileNotFoundError('Attach the CRNN-v2 package as a Kaggle input dataset.')
    extracted = WORK_ROOT / 'ocr_crnn_hand_v2_input'
    extracted.mkdir(exist_ok=True)
    with zipfile.ZipFile(zip_candidates[0]) as archive:
        archive.extractall(extracted)
    config_candidates = list(extracted.rglob('configs/ocr_crnn_hand_v2.json'))
CONFIG = config_candidates[0]
PACKAGE_ROOT = CONFIG.parents[1]
DATA_ROOT = PACKAGE_ROOT / 'data'
RUN_DIR = WORK_ROOT / 'ocr_crnn_hand_v2'
sys.path.insert(0, str(PACKAGE_ROOT))
print('Package:', PACKAGE_ROOT)
print('Data:', DATA_ROOT)
print('Output:', RUN_DIR)

In [ ]:
import cv2, numpy as np, torch
from src.vision.ocr_v2.package import verify_checksum_file

if not torch.cuda.is_available():
    raise RuntimeError('GPU is not enabled. In Kaggle notebook settings, select a GPU accelerator.')
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)
print('OpenCV:', cv2.__version__)
print('NumPy:', np.__version__)
print('Checksums:', verify_checksum_file(PACKAGE_ROOT))
print(json.loads((DATA_ROOT / 'dataset_report.json').read_text(encoding='utf-8')))

In [ ]:
train_command = [
    sys.executable, '-m', 'src.vision.ocr_v2.train',
    '--config', str(CONFIG),
    '--data-root', str(DATA_ROOT),
    '--output', str(RUN_DIR),
    '--device', 'cuda',
]
subprocess.run(train_command, cwd=PACKAGE_ROOT, check=True)

In [ ]:
from src.vision.ocr_v2.evaluate import metrics_summary
EVAL_DIR = RUN_DIR / 'independent_eval'
eval_command = [
    sys.executable, '-m', 'src.vision.ocr_v2.evaluate',
    '--checkpoint', str(RUN_DIR / 'best.pt'),
    '--manifest', str(DATA_ROOT / 'val_manifest.csv'),
    '--output', str(EVAL_DIR),
    '--device', 'cuda',
]
subprocess.run(eval_command, cwd=PACKAGE_ROOT, check=True)
metrics = json.loads((EVAL_DIR / 'metrics.json').read_text(encoding='utf-8'))
print(json.dumps(metrics_summary(metrics), ensure_ascii=False, indent=2))

In [ ]:
RESULT_ZIP = WORK_ROOT / 'ocr_crnn_hand_v2_results.zip'
result_base = RESULT_ZIP.with_suffix('')
result_zip = Path(shutil.make_archive(str(result_base), 'zip', root_dir=RUN_DIR))
print('Download this file from the Kaggle Output panel:')
print(result_zip, result_zip.stat().st_size, 'bytes')